# Two-stage comparison across time-frequency decompositions

Compares two-stage (N-vs-tremor → PD-vs-ET) classifiers built on different
**time-frequency decompositions**, chosen by the model-free separability study:
STFT-256 (best 3-class), HHT-8IMF (best PD-vs-ET), CWT, and a **hybrid**
(STFT-256 for stage 1, HHT-8IMF for stage 2). Honest leave-one-patient-out with
subject bootstrap CIs. CPU-only.

Note: HHT feature extraction takes ~1 min (transform is slow, but computed
once).

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd())=="pdetn": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from tremor.data import CLASS_NAMES
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.separability import patient_decomp_features
from pdetn.model import TwoStageClassifier, FlatClassifier, HybridTwoStage
from pdetn.evaluate import evaluate, evaluate_hybrid, print_result
DATA_ROOT="Data"; ACTION="OUT"
recs = load_quaternion_recordings(DATA_ROOT, action=ACTION, mode="angular_velocity")
print(len(recs), "recordings,", len(set(r.subject for r in recs)), "patients")

## 2. Per-patient decomposition features\nEach method reduced to mean+std over time, aggregated per patient. HHT is the slow one (~1 min).

In [ ]:
feat = {
  "stft256": patient_decomp_features(recs,"stft",nperseg=256,nfft=256,noverlap=192),
  "cwt":     patient_decomp_features(recs,"cwt",cwt_w0=6.0,cwt_freq_step=0.5),
  "hht8":    patient_decomp_features(recs,"hht",hht_max_imfs=8),
}
pats = feat["stft256"][2]; y = feat["stft256"][1]; subj = pats
assert all((feat[k][2]==pats).all() for k in feat)
print("features:", {k:v[0].shape for k,v in feat.items()})

## 3. Two-stage per single decomposition (tuned ET threshold)

In [ ]:
results={}
for name,(X,_,_) in feat.items():
    r=evaluate(lambda: TwoStageClassifier("logreg","logreg",tune_et_threshold=True), X,y,subj,n_boot=1000)
    results[f"2stage_{name}"]=r; print_result(f"2stage {name}", r)

## 4. Hybrid two-stage: STFT-256 (N-vs-tremor) + HHT-8IMF (PD-vs-ET)\nUse each representation where it separates best.

In [ ]:
X1=feat["stft256"][0]; X2=feat["hht8"][0]
rh=evaluate_hybrid(lambda: HybridTwoStage("logreg","logreg",tune_et_threshold=True), X1,X2,y,subj,n_boot=1000)
results["HYBRID_stft256+hht8"]=rh; print_result("HYBRID stft256->hht8", rh)

## 5. Comparison table + chart

In [ ]:
ref = {"deep_STFT(ref)":{"macro_f1":0.63,"per_class_f1":{"ET":0.47}},
       "biomarker_2stage(ref)":{"macro_f1":0.582,"per_class_f1":{"ET":0.324}}}
print(f"{'config':>24}{'macroF1':>9}{'ET_F1':>8}{'PDvsET':>8}{'Nvstre':>8}")
for k,r in results.items():
    print(f"{k:>24}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}{r.get('pd_vs_et_acc',float('nan')):>8.2f}{r.get('n_vs_tremor_acc',float('nan')):>8.2f}")
for k,r in ref.items(): print(f"{k:>24}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}")
labels=list(results)+list(ref); macro=[results.get(k,ref.get(k))['macro_f1'] for k in labels]
etf1=[(results.get(k,ref.get(k)))['per_class_f1']['ET'] for k in labels]
xi=np.arange(len(labels)); w=0.4
fig,ax=plt.subplots(figsize=(11,4)); ax.bar(xi-w/2,macro,w,label="macro-F1",color="#2c7fb8"); ax.bar(xi+w/2,etf1,w,label="ET-F1",color="#d95f02")
ax.set_xticks(xi); ax.set_xticklabels(labels,rotation=30,ha="right"); ax.axhline(1/3,ls="--",c="gray",lw=1,label="chance")
ax.set_ylabel("F1"); ax.set_title("Two-stage across TF decompositions (OUT)"); ax.legend(); plt.tight_layout(); plt.show()

## 6. Confusion matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
keys=list(results); fig,ax=plt.subplots(1,len(keys),figsize=(4*len(keys),3.6))
for a,k in zip(np.atleast_1d(ax),keys):
    ConfusionMatrixDisplay(np.array(results[k]['confusion_matrix']),display_labels=CLASS_NAMES).plot(ax=a,colorbar=False)
    a.set_title(f"{k}\nmF1 {results[k]['macro_f1']:.2f}",fontsize=9)
plt.tight_layout(); plt.show()

## 6.5 Temporal-spatial features (3-sensor arm geometry)

Add the spatial axis — hand/lower_arm/upper_arm tremor propagation (per-sensor
power, distal→proximal gradients, cross-sensor coherence & phase) — on top of
the time-frequency transform. Compare TF vs spatial vs **TF+spatial**, per
condition. Result: TF+spatial on OUT gives the best ET-F1 (0.42); WING gives the
best macro-F1 (0.67); spatial rescues the TF-poor REST condition.

In [ ]:
from collections import defaultdict
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.separability import patient_decomp_features
from pdetn.spatial_features import spatial_features, SPATIAL_FEATURE_NAMES
def patient_spatial(recs):
    per=defaultdict(list); lab={}
    for r in recs:
        per[r.subject].append([spatial_features(r.x)[f] for f in SPATIAL_FEATURE_NAMES]); lab[r.subject]=r.y
    pats=sorted(per); return (np.array([np.mean(per[p],axis=0) for p in pats]),
                              np.array([lab[p] for p in pats]), np.array(pats))
print(f"{'cond':>5}{'featureset':>14}{'macroF1':>9}{'ET_F1':>8}{'PDvsET':>8}")
for cond in ["OUT","REST","WING"]:
    rc=load_quaternion_recordings(DATA_ROOT,action=cond,mode="angular_velocity")
    Xtf,y,subj=patient_decomp_features(rc,"stft",nperseg=256,nfft=256,noverlap=192)
    Xsp,_,_=patient_spatial(rc)
    for nm,X in [("TF",Xtf),("spatial",Xsp),("TF+spatial",np.concatenate([Xtf,Xsp],1))]:
        r=evaluate(lambda: TwoStageClassifier("logreg","logreg",tune_et_threshold=True), X,y,subj,n_boot=500)
        print(f"{cond:>5}{nm:>14}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}{r['pd_vs_et_acc']:>8.2f}")

## 8. Quaternion-aware time-frequency (orientation-preserving)

Everything above reduces each sensor to a **scalar** spectrum, which throws away
the relative phase *between* the three axes — i.e. the shape and handedness of
the 3-D orbit the limb traces at each tremor frequency. That is exactly the kind
of information that could still separate PD from ET, because we have shown the
classes overlap ~70% along the **frequency** axis in two independent cohorts.

Four representations, per `pdetn/quaternion_tf.py`:

| representation | what it keeps | mount-invariant? |
|---|---|---|
| `omega` angular velocity + STFT | frequency/amplitude only | yes |
| `log_map` so(3) rotation vector + STFT | relative pose trajectory | yes (median-referenced) |
| **polarization** circularity / planarity | cross-axis phase, orbit *shape* | **yes** (invariant contractions of Z) |
| **QSTFT** simplex/perplex + chirality | orbit *handedness* | no — depends on the axis `mu` |
| **gravity-referenced chirality** | signed handedness vs gravity | **yes** |

Note on the log map: an *absolute* log map encodes how the sensor was strapped
on, which is subject-specific nuisance. `mode='log_map'` therefore references
each recording to its own median orientation, so a fixed re-mounting rotation
cancels.

In [ ]:
from pdetn.quaternion_tf import (polarization_features, qstft_features,
                                gravity_chirality_features, polarization_stft)
from pdetn.quaternion_repr import (load_repr, patient_table, patient_gchir_table,
                                   align, univariate_screen)

# Sanity check on synthetic orbits: does 'circularity' mean what we claim?
fs, T = 100.0, 1000; tt = np.arange(T)/fs
ccw = np.stack([np.cos(2*np.pi*6*tt),  np.sin(2*np.pi*6*tt), np.zeros(T)])
cw  = np.stack([np.cos(2*np.pi*6*tt), -np.sin(2*np.pi*6*tt), np.zeros(T)])
lin = np.stack([np.cos(2*np.pi*6*tt), 0.7*np.cos(2*np.pi*6*tt), np.zeros(T)])
g   = np.tile(np.array([[0.],[0.],[-1.]]), (1, T))
for nm, x in [("circular CCW", ccw), ("circular CW", cw), ("linear", lin)]:
    p = polarization_features(x, fs=fs); q = gravity_chirality_features(x, g, fs=fs)
    print(f"{nm:13s} circularity={p['circ_peak']:.3f}  signed handedness={q['gchir_peak']:+.3f}")

### 8.1 Which orbit-geometry features separate PD from ET?

Univariate screen (Mann-Whitney, rank-biserial effect). **Screening only** —
p-values are uncorrected and nothing here is used to select classifier features.

In [ ]:
_ = univariate_screen(data_root=DATA_ROOT, action=ACTION, top=14)

### 8.2 Do they improve the classifier?

Same two-stage LOSO, same tuned ET threshold, same subject bootstrap CI as
sections 3–4, so these rows are directly comparable to the table above.
Takes a few minutes (the 702-feature STFT sets dominate).

In [ ]:
from pdetn.quaternion_repr import compare
quat_results = compare(data_root=DATA_ROOT, action=ACTION, n_boot=1000, n_perm=1000)
results.update({f"quat_{k}": v for k, v in quat_results.items()})

### 8.3 Inspect an orbit map

The circularity map shows, per time-frequency cell, whether the limb is moving
along a line or around an ellipse. Compare a PD and an ET patient.

In [ ]:
om = load_repr(DATA_ROOT, ACTION, mode="angular_velocity")
pick = {}
for r in om:
    pick.setdefault(r.y, r)
fig, axes = plt.subplots(2, len(pick), figsize=(5*len(pick), 6))
for col, (cls, r) in enumerate(sorted(pick.items())):
    m = polarization_stft(r.x[6:9], fs=100.0)          # upper arm
    for row, key in enumerate(["power", "circularity"]):
        a = axes[row, col]
        a.pcolormesh(m["times"], m["freqs"], m["power"] if row == 0
                     else m["circularity"], shading="auto",
                     vmin=None if row == 0 else 0, vmax=None if row == 0 else 1)
        a.set_title(f"{CLASS_NAMES[cls]} — {key}", fontsize=9)
        a.set_ylabel("Hz")
axes[1, 0].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

### 8.4 The hybrid the comparison points to

`logmap_stft` is the best **stage-1** representation (N-vs-tremor 0.914) and the
worst stage-2; `grav_chirality` is the exact reverse. `HybridTwoStage` takes a
different feature matrix per stage, so use each where it wins.

Reported with **balanced** PD-vs-ET accuracy: with 75 PD and 15 ET the
majority-class baseline is 0.833, so raw accuracy on that axis is misleading.

In [ ]:
from pdetn.model import HybridTwoStage
from pdetn.evaluate import evaluate_hybrid
from pdetn.quaternion_repr import patient_gchir_table, load_repr, patient_table, align

om_r = load_repr(DATA_ROOT, ACTION, mode="angular_velocity")
lm_r = load_repr(DATA_ROOT, ACTION, mode="log_map")
gr_r = load_repr(DATA_ROOT, ACTION, mode="gravity")
skw  = dict(nperseg=256, nfft=256, noverlap=192)

X_base, y_h, pats_h = patient_decomp_features(om_r, "stft", **skw)[:3]
X_lmap = patient_decomp_features(lm_r, "stft", **skw)[0]
gch_t  = patient_gchir_table(om_r, gr_r)
geom_t = align(patient_table(om_r, block="polar"),
               patient_table(om_r, block="qstft"), gch_t)

n_pd, n_et = int((y_h == 1).sum()), int((y_h == 2).sum())
print(f"PD={n_pd} ET={n_et} -> PD-vs-ET majority baseline {n_pd/(n_pd+n_et):.3f}\n")

for name, (X1, X2) in {
    "S1 logmap -> S2 gchir":    (X_lmap, gch_t[0]),
    "S1 logmap -> S2 geometry": (X_lmap, geom_t[0]),
    "S1 omega  -> S2 gchir":    (X_base, gch_t[0]),
    "S1 omega  -> S2 omega":    (X_base, X_base),
}.items():
    r = evaluate_hybrid(lambda: HybridTwoStage("logreg", "logreg", tune_et_threshold=True),
                        X1, X2, y_h, pats_h, n_boot=1000, n_perm=1000)
    cm = np.array(r["confusion_matrix"])
    pd_r = cm[1,1]/max(cm[1,1]+cm[1,2],1); et_r = cm[2,2]/max(cm[2,1]+cm[2,2],1)
    ci = r["ci"]["ET"]
    results[f"hybrid_{name}"] = r
    print(f"{name:>26} macroF1 {r['macro_f1']:.3f} ET-F1 {r['per_class_f1']['ET']:.3f} "
          f"[{ci['lo']:.2f},{ci['hi']:.2f}] N-vs-T {r['n_vs_tremor_acc']:.3f} "
          f"PD-vs-ET bal-acc {0.5*(pd_r+et_r):.3f} p={r['permutation_p']:.4f}")

## 8.5 All three cohorts — LOCAL (2015), NewData (2025), PADS

Three cohorts, and they do **not** have the same capabilities:

| cohort | sensors | signal | classes | ET | orbit geometry? | log map / gravity? |
|---|---|---|---|---|---|---|
| LOCAL 2015 | 3 (hand/lower/upper arm) | quaternion | N/PD/ET | 15 | yes | yes |
| NewData 2025 | 3, both limbs | quaternion | **ET only** | 6 | yes | yes |
| PADS | 1 wrist | **gyro only** | N/PD/ET | 41 | yes | **no** |

PADS ships raw gyroscope with no orientation, so gravity-referenced chirality
and the log map cannot be computed on it — only the polarization/QSTFT features,
which need nothing but a 3-axis rate signal. That portability is itself a
finding: orbit *shape* transfers to any gyro dataset, orbit *handedness* needs
orientation.

**Run the device-identity probe before pooling anything.** NewData is ET-only,
so if its device is identifiable the model can learn "new device ⇒ ET" and
report a fake ET-F1 gain.

In [ ]:
from pdetn.load_2025 import load_2025
from pdetn.crossdataset import load_pads_extracted
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score

def device_probe(Xa, ga, Xb, gb, tag):
    """Can a classifier tell the two cohorts apart? AUC ~0.5 = safe to pool."""
    X = np.vstack([Xa, Xb]); d = np.r_[np.zeros(len(Xa)), np.ones(len(Xb))]
    g = np.r_[ga, gb]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    p = cross_val_predict(clf, X, d, groups=g, cv=LeaveOneGroupOut(),
                          method="predict_proba")[:, 1]
    auc = roc_auc_score(d, p)
    print(f"{tag:>46} AUC {auc:.3f}  {'CONFOUNDED' if auc > 0.75 else 'safe to pool'}")
    return auc

new_om = load_2025(mode="angular_velocity")          # ET only, 6 patients
new_gr = load_2025(mode="gravity")
new_gch = patient_gchir_table(new_om, new_gr)
new_geom = align(patient_table(new_om, block="polar"),
                 patient_table(new_om, block="qstft"), new_gch)
new_tf = patient_decomp_features(new_om, "stft", **skw)[:3]

print("device-identity probe, ET subjects only (LOCAL-ET vs NewData-ET):")
et = y_h == 2
device_probe(X_base[et], pats_h[et], new_tf[0],  new_geom[2], "stft-702")
device_probe(geom_t[0][et], pats_h[et], new_geom[0], new_geom[2], "orbit geometry-66")
device_probe(gch_t[0][et],  pats_h[et], new_gch[0],  new_gch[2],  "gravity-chirality-15")

### 8.6 Pooling NewData — only on the block that passes the probe

In [ ]:
from sklearn.metrics import f1_score, recall_score
from tremor.stats import bootstrap_subject_ci

def pd_vs_et(X, y, g, tag):
    m = y != 0
    X, yy, gg = X[m], (y[m] == 2).astype(int), g[m]
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=5000, class_weight="balanced"))
    p = cross_val_predict(clf, X, yy, groups=gg, cv=LeaveOneGroupOut(),
                          method="predict_proba")[:, 1]
    pr = (p >= 0.5).astype(int)
    bal = 0.5*(recall_score(yy, pr, pos_label=1) + recall_score(yy, pr, pos_label=0))
    e = bootstrap_subject_ci(yy, pr, gg, ["PD", "ET"], n_boot=2000, seed=0)["ET"]
    print(f"{tag:>38} PD={int((yy==0).sum()):>3} ET={int((yy==1).sum()):>3} "
          f"AUC {roc_auc_score(yy,p):.3f} bal-acc {bal:.3f} "
          f"ET-F1 {f1_score(yy,pr):.3f} [{e.lo:.2f},{e.hi:.2f}]")

print("gravity-referenced chirality (the block that passed the probe):")
pd_vs_et(gch_t[0], y_h, pats_h, "LOCAL only (15 ET)")
for sides, nm in [(("right",), "+NewData right"), (("left",), "+NewData left"),
                  (("right","left"), "+NewData both limbs")]:
    om_s = load_2025(mode="angular_velocity", sides=sides)
    gr_s = load_2025(mode="gravity", sides=sides)
    t = patient_gchir_table(om_s, gr_s)
    pd_vs_et(np.vstack([gch_t[0], t[0]]), np.r_[y_h, t[1]],
             np.r_[pats_h, t[2]], nm + " (21 ET)")

### 8.7 PADS — the independent replication test

PADS has 41 ET subjects, so its CIs are tight. It is a wrist sensor, so compare
against the LOCAL **lower_arm** (wrist-equivalent) channels only, and use the
orbit-geometry features, the only block PADS can support.

In [ ]:
from collections import defaultdict
from pdetn.quaternion_tf import (polarization_features, qstft_features,
                                 POLARIZATION_FEATURE_NAMES, QSTFT_FEATURE_NAMES)
from scipy.stats import mannwhitneyu

GEOM_COLS = POLARIZATION_FEATURE_NAMES + QSTFT_FEATURE_NAMES
def geom_table(recs, ch):
    per, lab = defaultdict(list), {}
    for r in recs:
        d = polarization_features(r.x[ch], fs=100.0)
        d.update(qstft_features(r.x[ch], fs=100.0))
        per[r.subject].append([d[c] for c in GEOM_COLS]); lab[r.subject] = r.y
    p = sorted(per)
    return (np.nan_to_num(np.array([np.mean(per[k], 0) for k in p])),
            np.array([lab[k] for k in p]), np.array(p))

pads = load_pads_extracted("pads_stretchhold")
Xp, yp, gp = geom_table(pads, [0, 1, 2])       # PADS single wrist
Xl, yl, gl = geom_table(om_r, [3, 4, 5])       # LOCAL lower_arm ~ wrist

pd_vs_et(Xl, yl, gl, "LOCAL lower_arm, orbit geometry")
pd_vs_et(Xp, yp, gp, "PADS wrist, orbit geometry")
device_probe(Xl[yl == 2], gl[yl == 2], Xp[yp == 2], gp[yp == 2],
             "LOCAL-ET vs PADS-ET")

print("\nfeature-level replication (PD vs ET in each cohort independently):")
print(f"{'feature':>20}{'LOCAL eff':>11}{'LOCAL p':>10}{'PADS eff':>11}{'PADS p':>10}")
for j, c in enumerate(GEOM_COLS):
    def st(X, y):
        a, b = X[y == 1, j], X[y == 2, j]
        u, p = mannwhitneyu(a, b); return 2*u/(len(a)*len(b)) - 1, p
    el, pl = st(Xl, yl); ep, pp = st(Xp, yp)
    if pl < 0.05 or pp < 0.05:
        tag = "  <-- replicates" if (pl < 0.05 and pp < 0.05 and el*ep > 0) else ""
        print(f"{c:>20}{el:>11.3f}{pl:>10.4f}{ep:>11.3f}{pp:>10.4f}{tag}")

## 9. Deep models — **trained here**

Sections 3–8 fit scikit-learn models on precomputed features; no network was
trained. This section actually trains BiLSTMs on the STFT images, under
**patient-level grouped CV** (every patient tested exactly once, no leakage),
with per-recording probabilities aggregated to patient level — so the rows land
in the same table as the classical results.

CPU-only and slow: budget ~10–25 min for the settings below. Lower `epochs` or
`n_splits` for a quick pass.

In [ ]:
from pdetn.deep_eval import deep_grouped_cv, multi_seed, pretrain_stage

recs_deep = load_quaternion_recordings(DATA_ROOT, action=ACTION, mode="angular_velocity")

# --- fine-tuning knobs -----------------------------------------------------
TRAIN_KW = dict(
    epochs=40, patience=10,
    lr=1e-3, weight_decay=1e-4,        # optimiser
    hidden=128, dropout=0.4,           # capacity
    focal_gamma=1.5,                   # class-imbalance emphasis
    tfd_method="stft", nperseg=256, noverlap=192,   # input representation
)
res_deep = deep_grouped_cv(recs_deep, mode="two_stage", n_splits=5,
                           tune_et=True, **TRAIN_KW)
print_result("deep two-stage (BiLSTM, trained here)", res_deep)
results["deep_two_stage"] = res_deep

### 9.1 One run is a draw, not a result

A single deep run varies by seed. Earlier in this project a single run read
0.903 and the 4-seed mean was 0.866 — so report the spread, not the best draw.

In [ ]:
# ~4x the cost of the cell above.
deep_seeds = multi_seed(recs_deep, seeds=(0, 1, 2, 3), mode="two_stage",
                        n_splits=5, tune_et=True, n_boot=200, n_perm=200,
                        **TRAIN_KW)

### 9.2 Fine-tuning from a pretrained stage

`pretrain_stage` trains one stage on an **external** cohort (e.g. PADS) and
returns its weights; `deep_grouped_cv(pretrain=...)` then warm-starts every fold
from those weights instead of random init. The external cohort is used only for
initialisation and never for scoring.

Requires the extracted PADS recordings — see `pdetn/README_PADS.md`.

In [ ]:
# from pdetn.crossdataset import load_pads_extracted
# pads = load_pads_extracted()                      # external cohort
# init = pretrain_stage(pads, stage="s2", epochs=40, **{k: v for k, v in TRAIN_KW.items()
#                                                       if k not in ("epochs", "patience")})
# res_ft = deep_grouped_cv(recs_deep, mode="two_stage", n_splits=5,
#                          pretrain=init, **TRAIN_KW)
# print_result("deep two-stage, PD-vs-ET stage fine-tuned from PADS", res_ft)
# results["deep_two_stage_finetuned"] = res_ft

### 9.3 Final comparison — classical vs quaternion-aware vs deep

In [ ]:
print(f"{'config':>32}{'macroF1':>9}{'ET_F1':>8}{'N-vs-T':>9}{'PD-vs-ET':>10}{'p':>8}")
for k, r in results.items():
    print(f"{k:>32}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}"
          f"{r.get('n_vs_tremor_acc', float('nan')):>9.3f}"
          f"{r.get('pd_vs_et_acc', float('nan')):>10.3f}"
          f"{r.get('permutation_p', float('nan')):>8.4f}")

## 7. Findings (see reports/decomposition_study.md)

- **STFT-256 two-stage is best overall** (macro-F1 0.651, ET-F1 0.378) — tuning the STFT window (256 vs default 128) gave a real gain over the biomarker feature set (0.582/0.324).
- **HHT-8 / hybrid win PD-vs-ET *accuracy*** (0.78–0.79, matching the separability study) but lower ET-F1 — ET-F1 also depends on stage-1 routing and the minority threshold.
- Decomposition study (model-free): STFT-256 best 3-class, HHT-7/8 best PD-vs-ET, CWT all-rounder; SST and feature fusion do not beat them.
- Ceiling is still ~16 ET subjects — external data (PADS) remains the real lever.